In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import os

In [57]:
df=pd.read_csv("fulfillment_raw.csv")

In [58]:
print(df.shape)

(512, 6)


In [59]:
print(df.head())

   order_id  order_date  warehouse     status  units  fulfillment_cost
0      5001  2024-03-23    WH-West        NaN    1.0             82.60
1      5002  2024-05-17    WH-East  Cancelled    1.0            459.37
2      5003  2024-02-24   WH-North    Delayed    4.0             81.43
3      5004  2024-01-24    WH-West  Delivered    5.0            105.71
4      5005  2024-02-27   WH-South  Cancelled    1.0            309.70


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 512 entries, 0 to 511
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          512 non-null    int64  
 1   order_date        512 non-null    object 
 2   warehouse         512 non-null    object 
 3   status            430 non-null    object 
 4   units             423 non-null    float64
 5   fulfillment_cost  512 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 24.1+ KB


In [61]:
print(df.isnull().sum())

order_id             0
order_date           0
warehouse            0
status              82
units               89
fulfillment_cost     0
dtype: int64


In [62]:
print(df.duplicated().sum())

12


In [63]:
df = df.drop_duplicates().copy()

In [64]:
df['warehouse'] = df['warehouse'].str.strip().str.upper()

In [65]:
df['status'] = df['status'].fillna('Unknown')

In [66]:
df['units'] = df['units'].fillna(df['units'].median())

In [67]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [68]:
print(df.isnull().sum())

order_id            0
order_date          0
warehouse           0
status              0
units               0
fulfillment_cost    0
dtype: int64


In [69]:
print(df.duplicated().sum())

0


In [70]:
print("Cleaned shape:", df.shape)

Cleaned shape: (500, 6)


In [71]:
print(df['warehouse'].unique())

['WH-WEST' 'WH-EAST' 'WH-NORTH' 'WH-SOUTH']


In [72]:
print(df.shape)

(500, 6)


In [73]:
conn = sqlite3.connect('fulfillment.db')

In [74]:
df.to_sql('fulfillment', conn, if_exists ='replace',index=False)

500

In [75]:
check_df = pd.read_sql("SELECT * FROM fulfillment LIMIT5",conn)

In [76]:
clean_df.to_csv('fulfillment_clean.csv', index=False)

NameError: name 'clean_df' is not defined

In [77]:
print("Exported successfully")

Exported successfully


In [78]:
print("Final shape:", clean_df.shape)

NameError: name 'clean_df' is not defined

In [79]:
print("Saved to:", os.getcwd())

Saved to: C:\Users\HP


In [80]:
query1 = """
SELECT warehouse,
COUNT (*) as Total_orders,
SUM(fulfillment_cost) as Total_cost,
ROUND(AVG(units),2) as Avg_units
FROM fulfillment
group by warehouse
order by Total_cost DESC
"""

In [81]:
print(pd.read_sql(query1, conn))

  warehouse  Total_orders  Total_cost  Avg_units
0  WH-SOUTH           172    47211.09       2.91
1  WH-NORTH           158    40488.59       3.03
2   WH-EAST            93    27605.74       3.00
3   WH-WEST            77    18647.30       3.03


In [50]:
query2 = """SELECT order_id, warehouse, fulfillment_cost,
       RANK() OVER (PARTITION BY warehouse ORDER BY fulfillment_cost DESC) as cost_rank
FROM fulfillment
ORDER BY warehouse, cost_rank
LIMIT 15
"""

In [51]:
print(pd.read_sql(query2, conn))

    order_id warehouse  fulfillment_cost  cost_rank
0       5360   WH-EAST            499.98          1
1       5143   WH-EAST            489.33          2
2       5226   WH-EAST            489.00          3
3       5370   WH-EAST            487.74          4
4       5099   WH-EAST            487.68          5
5       5458   WH-EAST            485.93          6
6       5264   WH-EAST            485.21          7
7       5187   WH-EAST            480.96          8
8       5418   WH-EAST            475.92          9
9       5275   WH-EAST            474.28         10
10      5133   WH-EAST            472.95         11
11      5304   WH-EAST            467.83         12
12      5126   WH-EAST            462.45         13
13      5002   WH-EAST            459.37         14
14      5426   WH-EAST            457.06         15


In [52]:
query3 = """
SELECT warehouse,
       status,
       COUNT(*) as order_count
FROM fulfillment
GROUP BY warehouse, status
ORDER BY warehouse, order_count DESC
"""
print(pd.read_sql(query3, conn))

   warehouse     status  order_count
0    WH-EAST  Delivered           45
1    WH-EAST    Unknown           19
2    WH-EAST    Delayed           17
3    WH-EAST  Cancelled           12
4   WH-NORTH  Delivered           89
5   WH-NORTH    Unknown           25
6   WH-NORTH    Delayed           24
7   WH-NORTH  Cancelled           20
8   WH-SOUTH  Delivered           88
9   WH-SOUTH    Delayed           35
10  WH-SOUTH    Unknown           26
11  WH-SOUTH  Cancelled           23
12   WH-WEST  Delivered           36
13   WH-WEST    Delayed           18
14   WH-WEST  Cancelled           14
15   WH-WEST    Unknown            9


In [82]:
import pandas as pd
import sqlite3
import os

df = pd.read_csv('fulfillment_raw.csv')
df = df.drop_duplicates().copy()
df['warehouse'] = df['warehouse'].str.strip().str.upper()
df['status'] = df['status'].fillna('Unknown')
df['units'] = df['units'].fillna(df['units'].median())
df['order_date'] = pd.to_datetime(df['order_date'])

print("Cleaned shape:", df.shape)

conn = sqlite3.connect('fulfillment.db')
df.to_sql('fulfillment', conn, if_exists='replace', index=False)

clean_df = pd.read_sql('SELECT * FROM fulfillment', conn)
clean_df.to_csv('fulfillment_clean.csv', index=False)

print("Exported successfully")
print("Final shape:", clean_df.shape)
print("Saved to:", os.getcwd())

Cleaned shape: (500, 6)
Exported successfully
Final shape: (500, 6)
Saved to: C:\Users\HP
